# Mask R-CNN â€” Clean Publication Pipeline

**Correct pipeline for publication-quality Mask R-CNN:**

1. **Clean original images** â€” YOLO-crop detections from stitched microscopy
2. **SAM auto-annotation** â€” Segment Anything generates masks on clean originals
3. **Manual verification** â€” every mask inspected / corrected by a human
4. **Clean binary masks** â€” `0 = background`, `1 = object` (verified)
5. **Paired augmentation** â€” Albumentations applied to **both image AND mask together** via `transform(image=image, mask=mask)` so spatial transforms stay consistent

**Anti-overfit strategy** (51 train + 11 val source images â†’ ~10 K crops):
- Train / Val split by original source image (no data leakage)
- 2-phase training: frozen backbone â†’ full fine-tune
- Early stopping + ReduceLROnPlateau
- Strong spatial + colour augmentation (train only)
- Weight decay 0.005 + gradient clipping

**Dataset:** `MyDrive/mp-detect/data/yolo-crop-sam/`
(images/, masks/, annotations.json â€” masks are manually verified)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_data = '/content/drive/MyDrive/mp-detect/data/yolo-crop-sam'
assert os.path.isdir(drive_data), f"Dataset not found: {drive_data}"
print(f"\u2713 Drive mounted â€” dataset at {drive_data}")

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations pycocotools tqdm matplotlib

In [ ]:
import json
import random
import time
import shutil
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision import transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\u2713 All imports OK")

## 3. Embedded Functions

> **Key design:** `CropDataset.__getitem__` calls
> `self.transforms(image=image, bboxes=..., masks=..., class_labels=...)`
> so Albumentations applies **every spatial transform identically** to the
> image, bounding box, **and** the binary mask.  This is the *paired
> augmentation* step â€” the mask always matches the augmented image.

In [ ]:
# ============================================================================
# EMBEDDED FUNCTIONS â€” clean-pipeline / anti-overfit edition
# ============================================================================

NUM_CLASSES = 4                                # bg + fiber + film + fragment
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
YOLO_TO_MASKRCNN = {0: 1, 1: 2, 2: 3}         # YOLO cls â†’ Mask R-CNN cls


# -----------------------------------------------------------------------
#                              DATASET
# -----------------------------------------------------------------------
class CropDataset(Dataset):
    """Dataset of manually-verified SAM-annotated crops for Mask R-CNN.

    Expected directory layout:
        crops_dir/
        â”œâ”€â”€ images/          # clean original crops (PNG)
        â”œâ”€â”€ masks/           # binary masks  0=background, 1=object (PNG)
        â””â”€â”€ annotations.json # {filename: {class_id, class_name, ...}}

    Paired augmentation
    -------------------
    The Albumentations pipeline receives BOTH the image AND the mask via
        transformed = self.transforms(image=image, bboxes=..., masks=[mask], ...)
    so every spatial transform (flip, rotate, elastic, shift-scale-rotate, â€¦)
    is applied **identically** to the image and its mask.  Colour / noise
    transforms only modify the image channel â€” the mask stays binary.

    Args:
        crops_dir:    Root directory (must contain images/, masks/).
        transforms:   Albumentations Compose pipeline.
        split_filter: 'train' or 'val' â€” uses the 'split' key in annotations.
        sample_keys:  Explicit list of annotation keys to use (overrides split_filter).
    """

    CLASS_NAME_TO_YOLO_ID = {'fiber': 0, 'film': 1, 'fragment': 2}

    def __init__(self, crops_dir: str, transforms=None,
                 split_filter: str = None, sample_keys: list = None):
        self.crops_dir = Path(crops_dir)
        self.transforms = transforms

        ann_file = self.crops_dir / 'annotations.json'
        has_flat_images = (self.crops_dir / 'images').is_dir()
        has_class_dirs  = any((self.crops_dir / c).is_dir()
                              for c in self.CLASS_NAME_TO_YOLO_ID)

        # ---------- Load / build annotations ----------
        if ann_file.exists():
            with open(ann_file) as f:
                self.annotations = json.load(f)
            self.images_dir = (self.crops_dir / 'images') if has_flat_images else None
        elif has_class_dirs:
            self.annotations = {}
            self.images_dir = None
            for cls_name, cls_id in self.CLASS_NAME_TO_YOLO_ID.items():
                cls_dir = self.crops_dir / cls_name
                if not cls_dir.is_dir():
                    continue
                for img_file in sorted(cls_dir.glob('*.png')):
                    crop = cv2.imread(str(img_file))
                    if crop is None:
                        continue
                    h, w = crop.shape[:2]
                    self.annotations[img_file.name] = {
                        'source_image': '', 'class_id': cls_id,
                        'class_name': cls_name, 'yolo_confidence': 1.0,
                        'rel_box': [0, 0, w, h], 'crop_size': [w, h],
                    }
            print(f"Auto-generated annotations for {len(self.annotations)} crops")
        else:
            raise FileNotFoundError(
                f"Neither annotations.json nor class subdirs found in {crops_dir}")

        # ---------- Filter ----------
        if sample_keys is not None:
            self.samples = [k for k in sample_keys if k in self.annotations]
        elif split_filter:
            self.samples = [
                k for k, v in self.annotations.items()
                if v.get('split', 'train') == split_filter
            ]
        else:
            self.samples = list(self.annotations.keys())

        print(f"CropDataset: {len(self.samples)} samples"
              + (f" (split={split_filter})" if split_filter else ""))

    # ------------------------------------------------------------------ #
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        ann  = self.annotations[name]

        # --- Load image ---
        if self.images_dir is not None:
            img_path = self.images_dir / name
        else:
            img_path = self.crops_dir / ann.get('class_name', '') / name

        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        # --- Load verified binary mask ---
        mask = None
        mask_file = ann.get('mask_file')
        masks_dir = self.crops_dir / 'masks'

        if mask_file and (masks_dir / mask_file).exists():
            raw = cv2.imread(str(masks_dir / mask_file), cv2.IMREAD_GRAYSCALE)
            if raw is not None:
                mask = (raw > 127).astype(np.uint8)

        if mask is None:
            default_mask = masks_dir / name.replace('.png', '_mask.png')
            if default_mask.exists():
                raw = cv2.imread(str(default_mask), cv2.IMREAD_GRAYSCALE)
                if raw is not None:
                    mask = (raw > 127).astype(np.uint8)

        if mask is None:
            # Last resort â€” ellipse placeholder (should not happen with
            # verified data, but keeps the pipeline from crashing)
            mask = self._create_ellipse_mask(h, w, ann.get('rel_box'))

        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        # --- Bounding box from mask foreground ---
        ys, xs = np.where(mask > 0)
        if len(xs) > 0 and len(ys) > 0:
            box = [xs.min(), ys.min(), xs.max(), ys.max()]
        else:
            margin = min(h, w) // 10
            box = [margin, margin, w - margin, h - margin]

        class_id = YOLO_TO_MASKRCNN[ann['class_id']]
        boxes  = np.array([box], dtype=np.float32)
        labels = np.array([class_id], dtype=np.int64)
        masks  = np.array([mask], dtype=np.uint8)

        # ====== PAIRED AUGMENTATION ======
        # image + mask + bbox go through the SAME spatial transforms
        if self.transforms:
            transformed = self.transforms(
                image=image,
                bboxes=boxes.tolist(),
                masks=list(masks),           # list of 2-D numpy arrays
                class_labels=labels.tolist(),
            )
            image  = transformed['image']                       # tensor
            if len(transformed['bboxes']) > 0:
                boxes  = np.array(transformed['bboxes'], dtype=np.float32)
                labels = np.array(transformed['class_labels'], dtype=np.int64)
                masks  = np.array(transformed['masks'], dtype=np.uint8)
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        target = {
            'boxes':    torch.as_tensor(boxes,  dtype=torch.float32),
            'labels':   torch.as_tensor(labels, dtype=torch.int64),
            'masks':    torch.as_tensor(masks,  dtype=torch.uint8),
            'image_id': torch.tensor([idx]),
            'area':     torch.as_tensor(
                            [(b[2]-b[0])*(b[3]-b[1]) for b in boxes],
                            dtype=torch.float32),
            'iscrowd':  torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image, target

    @staticmethod
    def _create_ellipse_mask(h, w, rel_box=None):
        mask = np.zeros((h, w), dtype=np.uint8)
        if rel_box:
            x1, y1, x2, y2 = rel_box
            center = ((x1+x2)//2, (y1+y2)//2)
            axes   = ((x2-x1)//2, (y2-y1)//2)
        else:
            center = (w//2, h//2)
            axes   = (int(w*0.4), int(h*0.4))
        if axes[0] > 0 and axes[1] > 0:
            cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
        return mask


# -----------------------------------------------------------------------
#                      PAIRED  AUGMENTATION  PIPELINES
# -----------------------------------------------------------------------
def get_transforms(train=True, img_size=128):
    """Return Albumentations Compose with bbox + mask support.

    IMPORTANT â€” both the train AND val pipelines use
        A.BboxParams(...) and accept `masks=` in the __call__
    so spatial transforms (resize, flip, rotate, elastic, â€¦) are applied
    identically to image, mask, AND bounding box.

    Colour / noise transforms (brightness, hue, Gauss noise, blur, dropout)
    only affect the image tensor â€” the binary mask is untouched.
    """
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            # ---- spatial (applied to image + mask + bbox) ----
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                               rotate_limit=30, p=0.5),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            # ---- colour / noise (image only â€” mask unchanged) ----
            A.RandomBrightnessContrast(brightness_limit=0.3,
                                       contrast_limit=0.3, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10,
                                 sat_shift_limit=20,
                                 val_shift_limit=20, p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            # ---- cutout (image only) ----
            A.CoarseDropout(max_holes=4, max_height=img_size // 8,
                            max_width=img_size // 8, p=0.3),
            # ---- normalize + to tensor ----
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(
            format='pascal_voc', label_fields=['class_labels'],
            min_visibility=0.3))
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(
            format='pascal_voc', label_fields=['class_labels'],
            min_visibility=0.3))


# -----------------------------------------------------------------------
#                          MODEL  HELPERS
# -----------------------------------------------------------------------
def collate_fn(batch):
    return tuple(zip(*batch))


def get_model(num_classes: int, pretrained: bool = True):
    """Mask R-CNN ResNet50-FPN with custom head for `num_classes`."""
    if pretrained:
        model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    else:
        model = maskrcnn_resnet50_fpn(weights=None)
    in_f  = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor  = FastRCNNPredictor(in_f, num_classes)
    in_fm = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_fm, 256, num_classes)
    return model


def freeze_backbone(model, freeze=True):
    """Freeze / unfreeze the ResNet-50 + FPN backbone."""
    for p in model.backbone.parameters():
        p.requires_grad = not freeze
    status = "frozen" if freeze else "unfrozen"
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Backbone {status}  ({n:,} trainable params)")


@torch.no_grad()
def evaluate(model, loader, device):
    """Run model in train mode on val data to compute loss (no grad)."""
    model.train()
    total_loss = 0.0
    comp_losses = defaultdict(float)
    count = 0
    for images, targets in loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        if not all(len(t['boxes']) > 0 for t in targets):
            continue
        loss_dict = model(images, targets)
        total_loss += sum(v.item() for v in loss_dict.values())
        for k, v in loss_dict.items():
            comp_losses[k] += v.item()
        count += 1
    avg = total_loss / max(count, 1)
    comp = {k: v / max(count, 1) for k, v in comp_losses.items()}
    return avg, comp


print("\u2713 All functions defined (clean-pipeline / anti-overfit)")

## 4. Copy Dataset to Colab Local SSD

Copy `yolo-crop-sam/` from Google Drive to `/content/mp_data/` (fast local NVMe).
Resilient to Drive disconnects â€” copies file-by-file with retries and auto-remount.
Re-run this cell to **resume** if interrupted.

In [ ]:
# ============================================================================
# Resilient copy: Drive â†’ Local SSD  (file-by-file, retry + auto-remount)
# ============================================================================

# --- Paths ---
DRIVE_ROOT     = Path('/content/drive/MyDrive/mp-detect')
DRIVE_SAM_DIR  = DRIVE_ROOT / 'data' / 'yolo-crop-sam'

LOCAL_ROOT = Path('/content/mp_data')
SAM_DIR    = LOCAL_ROOT / 'yolo-crop-sam'
SAVE_DIR   = LOCAL_ROOT / 'experiments'

DRIVE_EXPERIMENTS = DRIVE_ROOT / 'experiments'


def _remount_drive():
    """Force-remount Google Drive after a disconnect."""
    print("  Remounting Google Drive ...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        time.sleep(2)
    except Exception:
        pass
    from google.colab import drive as _d
    _d.mount('/content/drive', force_remount=True)
    time.sleep(3)
    print("  Drive remounted")


def robust_copy_file(src: Path, dst: Path, max_retries=3):
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_retries + 1):
        try:
            shutil.copy2(str(src), str(dst))
            return True
        except OSError as e:
            if e.errno == 107 or 'Transport endpoint' in str(e):
                print(f"  Drive disconnect copying {src.name} "
                      f"(attempt {attempt}/{max_retries})")
                _remount_drive()
            else:
                raise
    print(f"  FAILED after {max_retries} retries: {src.name}")
    return False


def robust_copytree(src_dir: Path, dst_dir: Path):
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    copied, skipped, failed = 0, 0, 0
    files_only = [f for f in src_dir.rglob('*') if f.is_file()]
    print(f"  Found {len(files_only)} files to copy")
    for src_file in files_only:
        rel = src_file.relative_to(src_dir)
        dst_file = dst_dir / rel
        if dst_file.exists() and dst_file.stat().st_size > 0:
            skipped += 1
            continue
        ok = robust_copy_file(src_file, dst_file)
        if ok:
            copied += 1
        else:
            failed += 1
        total = copied + skipped + failed
        if total % 200 == 0:
            print(f"  ... {copied} copied, {skipped} skipped, "
                  f"{failed} failed / {len(files_only)}")
    print(f"  Done: {copied} copied, {skipped} existed, {failed} failed")
    return failed == 0


# --- Execute copy (resumable) ---
marker = SAM_DIR / '.copy_complete'
if not marker.exists():
    SAM_DIR.mkdir(parents=True, exist_ok=True)
    print("Copying verified SAM dataset to local storage ...")
    if robust_copytree(DRIVE_SAM_DIR, SAM_DIR):
        marker.touch()
        print(f"Copy complete -> {SAM_DIR}")
    else:
        print("Some files failed -- re-run this cell to resume")
else:
    print(f"SAM data already local: {SAM_DIR}")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# Hyperparameters
# ============================================================================
CROP_SIZE          = 128
MASKRCNN_BATCH     = 8

PHASE1_EPOCHS      = 5        # backbone frozen â€” warm up heads
PHASE1_LR          = 5e-4

PHASE2_EPOCHS      = 20       # full fine-tune
PHASE2_LR          = 1e-4     # 5x smaller

WEIGHT_DECAY       = 0.005    # aggressive L2
EARLY_STOP_PATIENCE = 7

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"\nDevice: {DEVICE}")
print(f"Data:   {SAM_DIR}")
print(f"Save:   {SAVE_DIR}")
print(f"\nPlan:")
print(f"  Phase 1: {PHASE1_EPOCHS} ep, LR={PHASE1_LR}, backbone FROZEN")
print(f"  Phase 2: {PHASE2_EPOCHS} ep, LR={PHASE2_LR}, backbone UNFROZEN")
print(f"  Early-stop patience: {EARLY_STOP_PATIENCE}")
print(f"  Weight decay: {WEIGHT_DECAY}")

## 5. Explore Verified Dataset

Sanity-check: count images, masks, class balance, and overlay a few masks.

In [ ]:
sam_path = Path(SAM_DIR)

with open(sam_path / 'annotations.json') as f:
    annotations = json.load(f)

n_images = len(list((sam_path / 'images').glob('*.png')))
n_masks  = len(list((sam_path / 'masks').glob('*.png')))
print(f"Images:      {n_images}")
print(f"Masks:       {n_masks}")
print(f"Annotations: {len(annotations)}")

# Check mask pairing
missing_masks = 0
for name, ann in annotations.items():
    mf = ann.get('mask_file', name.replace('.png', '_mask.png'))
    if not (sam_path / 'masks' / mf).exists():
        missing_masks += 1
print(f"Missing masks: {missing_masks}" +
      (" -- all paired" if missing_masks == 0 else " WARNING"))

# Class distribution
class_counts = defaultdict(int)
for ann in annotations.values():
    class_counts[ann.get('class_name', 'unknown')] += 1

print(f"\nClass distribution:")
for cls, count in sorted(class_counts.items()):
    pct = count / len(annotations) * 100
    print(f"  {cls:10s}: {count:5d} ({pct:.1f}%)")

# Show 6 random samples with verified masks
keys = random.sample(list(annotations.keys()), min(6, len(annotations)))
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle('Verified SAM Masks (top: original, bottom: mask overlay)',
             fontsize=14)

for j, name in enumerate(keys):
    ann = annotations[name]
    img = cv2.imread(str(sam_path / 'images' / name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mf = ann.get('mask_file', name.replace('.png', '_mask.png'))
    mp = sam_path / 'masks' / mf
    mask = None
    if mp.exists():
        raw = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        if raw is not None:
            mask = (raw > 127).astype(np.uint8)

    axes[0, j].imshow(img)
    axes[0, j].set_title(ann.get('class_name', '?'), fontsize=9)
    axes[0, j].axis('off')

    if mask is not None:
        overlay = img.copy()
        overlay[mask == 1] = (overlay[mask == 1] * 0.5 +
                              np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        axes[1, j].imshow(overlay)
    else:
        axes[1, j].imshow(img)
    axes[1, j].set_title('verified mask', fontsize=9)
    axes[1, j].axis('off')

plt.tight_layout()
plt.show()

## 6. Train / Val Split & Build Datasets

Split by **source_image** â€” crops from validation-origin images are never seen during training.

> Both `train_dataset` and `val_dataset` call the **same** `CropDataset`,
> which applies paired augmentation: `transform(image=image, masks=[mask], ...)`.
> Only the **train** pipeline adds spatial / colour jitter; val uses
> resize + normalize only.

In [ ]:
sam_path = Path(SAM_DIR)
with open(sam_path / 'annotations.json') as f:
    all_annotations = json.load(f)

# ---- Decide split strategy ----
has_split = any(v.get('split') for v in all_annotations.values())

if has_split:
    train_keys = [k for k, v in all_annotations.items()
                  if v.get('split') == 'train']
    val_keys   = [k for k, v in all_annotations.items()
                  if v.get('split') == 'val']
    print("Using annotation 'split' field")
else:
    source_to_keys = defaultdict(list)
    for k, v in all_annotations.items():
        source_to_keys[v.get('source_image', k)].append(k)
    sources = sorted(source_to_keys.keys())
    random.seed(42)
    random.shuffle(sources)
    n_train = max(1, int(len(sources) * 0.8))
    train_sources = set(sources[:n_train])
    val_sources   = set(sources[n_train:])
    train_keys = [k for s in train_sources for k in source_to_keys[s]]
    val_keys   = [k for s in val_sources   for k in source_to_keys[s]]
    print(f"No 'split' field -- split by source image "
          f"({len(train_sources)} train / {len(val_sources)} val sources)")

print(f"\nTrain crops: {len(train_keys)}")
print(f"Val   crops: {len(val_keys)}")

# Class balance per split
for label, keys in [('Train', train_keys), ('Val', val_keys)]:
    counts = defaultdict(int)
    for k in keys:
        counts[all_annotations[k].get('class_name', '?')] += 1
    print(f"\n  {label}:")
    for cls in sorted(counts):
        print(f"    {cls:10s}: {counts[cls]}")

# ---- Build datasets (paired augmentation) ----
train_tf = get_transforms(train=True,  img_size=CROP_SIZE)
val_tf   = get_transforms(train=False, img_size=CROP_SIZE)

train_dataset = CropDataset(str(SAM_DIR), transforms=train_tf,
                            sample_keys=train_keys)
val_dataset   = CropDataset(str(SAM_DIR), transforms=val_tf,
                            sample_keys=val_keys)

# Quick check
img, tgt = train_dataset[0]
print(f"\nSample -- shape: {img.shape}, "
      f"class: {CLASS_NAMES[tgt['labels'][0].item()]}, "
      f"mask sum: {tgt['masks'].sum().item()}")

# Visualise augmented training samples to confirm paired transforms
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Paired Augmentation Check  (image + mask transformed together)',
             fontsize=14)

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(8):
    idx = random.randint(0, len(train_dataset) - 1)
    img, tgt = train_dataset[idx]
    row, col = i // 4, i % 4
    disp = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    mask = tgt['masks'][0].numpy()
    overlay = disp.copy()
    overlay[mask == 1] = overlay[mask == 1] * 0.5 + np.array([0, 1, 0]) * 0.5
    axes[row, col].imshow(overlay)
    axes[row, col].set_title(
        f"{CLASS_NAMES[tgt['labels'][0].item()]}  "
        f"(mask px: {mask.sum()})", fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 7. Model & DataLoaders

In [ ]:
model = get_model(NUM_CLASSES, pretrained=True)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Mask R-CNN (ResNet50-FPN)")
print(f"  Parameters: {total_params:,}")
print(f"  Classes:    {NUM_CLASSES} {CLASS_NAMES}")
print(f"  Device:     {DEVICE}")

train_loader = DataLoader(train_dataset, batch_size=MASKRCNN_BATCH,
                          shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=MASKRCNN_BATCH,
                          shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"\n  Train: {len(train_dataset):,} samples -> {len(train_loader)} batches")
print(f"  Val:   {len(val_dataset):,} samples -> {len(val_loader)} batches")

## 8. Train (2-Phase + Early Stopping)

| Phase | Backbone | Epochs | LR | Purpose |
|-------|----------|--------|----|---------|
| 1 | **Frozen** | 5 | 5e-4 | Warm up detection heads safely |
| 2 | **Unfrozen** | 20 | 1e-4 | End-to-end fine-tune with early stopping |

In [ ]:
# ============================================================================
# 2-PHASE TRAINING  with val loss tracking + early stopping
# ============================================================================
best_val_loss    = float('inf')
patience_counter = 0
global_epoch     = 0

history = {
    'epoch': [], 'train_loss': [], 'val_loss': [], 'lr': [], 'phase': [],
    'loss_classifier': [], 'loss_box_reg': [],
    'loss_mask': [], 'loss_objectness': [], 'loss_rpn_box_reg': [],
    'val_loss_mask': [],
}


def run_phase(phase_name, n_epochs, lr, freeze_bb):
    global best_val_loss, patience_counter, global_epoch
    early_stopped = False

    freeze_backbone(model, freeze=freeze_bb)

    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True)

    print(f"\n{'='*60}")
    print(f"{phase_name}: {n_epochs} ep, LR={lr}, "
          f"backbone={'FROZEN' if freeze_bb else 'UNFROZEN'}")
    print(f"{'='*60}\n")

    for ep in range(1, n_epochs + 1):
        global_epoch += 1
        model.train()
        epoch_loss = 0.0
        epoch_comp = defaultdict(float)
        n_batch    = 0

        pbar = tqdm(train_loader,
                    desc=f"[{phase_name}] Ep {ep}/{n_epochs}")
        for images, targets in pbar:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()}
                       for t in targets]
            if not all(len(t['boxes']) > 0 for t in targets):
                continue

            loss_dict = model(images, targets)
            losses    = sum(v for v in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()

            bl = losses.item()
            epoch_loss += bl
            n_batch    += 1
            for k, v in loss_dict.items():
                epoch_comp[k] += v.item()
            pbar.set_postfix({'loss': f'{bl:.4f}'})

        avg_train = epoch_loss / max(n_batch, 1)

        # ---- Validation ----
        val_loss, val_comp = evaluate(model, val_loader, DEVICE)
        scheduler.step(val_loss)
        cur_lr = optimizer.param_groups[0]['lr']

        # ---- Record ----
        history['epoch'].append(global_epoch)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(val_loss)
        history['lr'].append(cur_lr)
        history['phase'].append(phase_name)
        for k in ['loss_classifier', 'loss_box_reg', 'loss_mask',
                   'loss_objectness', 'loss_rpn_box_reg']:
            history[k].append(epoch_comp.get(k, 0) / max(n_batch, 1))
        history['val_loss_mask'].append(val_comp.get('loss_mask', 0))

        print(f"  Ep {ep}/{n_epochs}  train={avg_train:.4f}  "
              f"val={val_loss:.4f}  val_mask={val_comp.get('loss_mask',0):.4f}  "
              f"LR={cur_lr:.6f}")

        # ---- Checkpoint ----
        ckpt = {
            'epoch':                global_epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss':           avg_train,
            'val_loss':             val_loss,
            'history':              history,
        }
        torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_latest.pth'))

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_best.pth'))
            print(f"    -> New best! (val_loss={val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"    Early stopping (patience {EARLY_STOP_PATIENCE})")
                early_stopped = True
                break

    return early_stopped


# ---- Phase 1: backbone FROZEN ----
stopped = run_phase("Phase 1 (heads)", PHASE1_EPOCHS, PHASE1_LR,
                    freeze_bb=True)

# ---- Phase 2: full fine-tune ----
if not stopped:
    run_phase("Phase 2 (full)", PHASE2_EPOCHS, PHASE2_LR,
              freeze_bb=False)

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE -- {global_epoch} epochs, best val = {best_val_loss:.4f}")
print(f"{'='*60}")

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Mask R-CNN Training Curves', fontsize=14)

epochs = history['epoch']

# ---- Total loss: train vs val ----
axes[0, 0].plot(epochs, history['train_loss'], 'b-', lw=2, label='Train')
axes[0, 0].plot(epochs, history['val_loss'],   'r-', lw=2, label='Val')
if PHASE1_EPOCHS < len(epochs):
    axes[0, 0].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5,
                        label='Unfreeze')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Total Loss'); axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ---- Component losses (train) ----
for key, color, label in [
    ('loss_classifier', 'r', 'Classifier'), ('loss_box_reg', 'g', 'Box Reg'),
    ('loss_mask', 'b', 'Mask'), ('loss_objectness', 'm', 'Objectness'),
    ('loss_rpn_box_reg', 'c', 'RPN Box Reg'),
]:
    if history.get(key):
        axes[0, 1].plot(epochs, history[key], color=color, label=label, lw=1.5)
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Train Components')
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)

# ---- Learning rate ----
axes[1, 0].plot(epochs, history['lr'], 'g-', lw=2)
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('LR')
axes[1, 0].set_title('LR (ReduceLROnPlateau)')
axes[1, 0].grid(True, alpha=0.3)

# ---- Mask loss: train vs val ----
if history.get('loss_mask') and history.get('val_loss_mask'):
    axes[1, 1].plot(epochs, history['loss_mask'],     'b-', lw=2, label='Train')
    axes[1, 1].plot(epochs, history['val_loss_mask'], 'r-', lw=2, label='Val')
    if PHASE1_EPOCHS < len(epochs):
        axes[1, 1].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5)
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Mask Loss')
    axes[1, 1].set_title('Mask Loss (primary metric)')
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ---- Overfit diagnostic ----
gap = history['val_loss'][-1] - history['train_loss'][-1]
print(f"\nFinal (Epoch {epochs[-1]}):")
print(f"  Train: {history['train_loss'][-1]:.4f}")
print(f"  Val:   {history['val_loss'][-1]:.4f}")
print(f"  Gap:   {gap:.4f} {'(OVERFIT WARNING)' if gap > 0.5 else '(OK)'}")
print(f"  Best:  {best_val_loss:.4f}")

## 10. Visualise Predictions vs Verified Ground Truth

In [ ]:
# Load best checkpoint
ckpt = torch.load(str(SAVE_DIR / 'maskrcnn_crops_best.pth'),
                  map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model -- epoch {ckpt['epoch']}, "
      f"val_loss {ckpt['val_loss']:.4f}")

inference_tf = T.Compose([
    T.ToPILImage(),
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

sam_path = Path(SAM_DIR)
with open(sam_path / 'annotations.json') as f:
    test_anns = json.load(f)

samples = random.sample(list(test_anns.keys()), min(8, len(test_anns)))

class_colors = {1: (255, 50, 50), 2: (50, 255, 50), 3: (50, 50, 255)}

fig, axes = plt.subplots(len(samples), 4,
                         figsize=(20, 4 * len(samples)))
if len(samples) == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Predictions vs Verified GT', fontsize=14, y=1.01)
for col, title in enumerate(['Original', 'Verified GT',
                              'Predicted Mask', 'Overlay']):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i, name in enumerate(samples):
    ann = test_anns[name]

    img = cv2.imread(str(sam_path / 'images' / name))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Col 0 -- Original
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_ylabel(ann['class_name'], fontsize=10,
                          rotation=0, labelpad=50)
    axes[i, 0].axis('off')

    # Col 1 -- Verified GT mask overlay
    mf = ann.get('mask_file', name.replace('.png', '_mask.png'))
    gtp = sam_path / 'masks' / mf
    if gtp.exists():
        gt = cv2.imread(str(gtp), cv2.IMREAD_GRAYSCALE)
        gt_bin = (gt > 127).astype(np.uint8)
        ov = img_rgb.copy()
        ov[gt_bin == 1] = (ov[gt_bin == 1] * 0.5 +
                           np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        axes[i, 1].imshow(ov)
    axes[i, 1].axis('off')

    # Inference
    inp = inference_tf(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(inp)[0]

    keep = pred['scores'] > 0.3
    if keep.sum() > 0:
        best_mask  = pred['masks'][keep][0, 0].cpu().numpy()
        best_label = pred['labels'][keep][0].item()
        best_score = pred['scores'][keep][0].item()
        best_box   = pred['boxes'][keep][0].cpu().numpy().astype(int)
        cls_name   = CLASS_NAMES[best_label] if best_label < len(CLASS_NAMES) else '?'

        # Col 2 -- Predicted mask
        axes[i, 2].imshow(best_mask > 0.5, cmap='gray')
        axes[i, 2].set_title(f"{cls_name} ({best_score:.2f})", fontsize=9)
        axes[i, 2].axis('off')

        # Col 3 -- Overlay + bbox
        img_r = cv2.resize(img_rgb, (CROP_SIZE, CROP_SIZE))
        pred_bin = (best_mask > 0.5).astype(np.uint8)
        color = class_colors.get(best_label, (255, 255, 0))
        c_norm = np.array(color) / 255.0
        ov = img_r.copy().astype(np.float32) / 255.0
        ov[pred_bin == 1] = ov[pred_bin == 1] * 0.5 + c_norm * 0.5
        ov_u8 = (ov * 255).astype(np.uint8)
        x1, y1, x2, y2 = best_box
        cv2.rectangle(ov_u8, (x1, y1), (x2, y2), color, 2)
        cv2.putText(ov_u8, f"{cls_name} {best_score:.2f}",
                    (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
        axes[i, 3].imshow(ov_u8)
        axes[i, 3].axis('off')
    else:
        axes[i, 2].text(0.5, 0.5, 'No detection',
                        ha='center', va='center', fontsize=12)
        axes[i, 2].axis('off')
        axes[i, 3].text(0.5, 0.5, 'No detection',
                        ha='center', va='center', fontsize=12)
        axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## 11. Save Models to Google Drive

In [ ]:
DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)

for mf in ['maskrcnn_crops_best.pth', 'maskrcnn_crops_latest.pth']:
    src = SAVE_DIR / mf
    dst = DRIVE_EXPERIMENTS / mf
    if src.exists():
        shutil.copy2(str(src), str(dst))
        mb = src.stat().st_size / 1024 / 1024
        print(f"{mf} ({mb:.1f} MB) -> Drive")
    else:
        print(f"{mf} not found locally")

print(f"\n{'='*60}")
print("MODELS SAVED TO GOOGLE DRIVE")
print(f"  Best:   {DRIVE_EXPERIMENTS / 'maskrcnn_crops_best.pth'}")
print(f"  Latest: {DRIVE_EXPERIMENTS / 'maskrcnn_crops_latest.pth'}")
print(f"{'='*60}")

## Done!

**Pipeline recap:**
```
Clean crops  ->  SAM masks  ->  Manual verification  ->  Paired augmentation  ->  Mask R-CNN
   (YOLO)         (auto)          (human QC)             (image + mask)          (this notebook)
```

**Models saved to:** `MyDrive/mp-detect/experiments/`
- `maskrcnn_crops_best.pth` â€” lowest validation loss
- `maskrcnn_crops_latest.pth` â€” last epoch

**Anti-overfit measures (51 + 11 source images -> ~10 K crops):**
- Train / Val split by source image (no leakage)
- Phase 1: backbone frozen (5 ep, LR 5e-4)
- Phase 2: full fine-tune (20 ep, LR 1e-4) + early stopping (patience 7)
- ReduceLROnPlateau (factor 0.5, patience 3)
- Strong paired augmentation: elastic, affine, HSV, dropout, blur
- Weight decay 0.005 + gradient clipping

**Key correctness guarantee:** all spatial augmentations (flip, rotate, elastic,
shift-scale-rotate) are applied to **both the image and the binary mask** via
`Albumentations(image=img, masks=[mask], bboxes=..., class_labels=...)` â€”
the mask always matches the augmented image pixel-for-pixel.

**Use in full pipeline:**
```bash
python src/pipeline_inference.py \
    --input dev-test/stitched/s1.png \
    --yolo experiments/augmented_microplastic_yolo/weights/best.pt \
    --effnet experiments/efficientnet_best.pth \
    --maskrcnn experiments/maskrcnn_crops_best.pth
```